In [16]:
import pandas as pd
import numpy as np
import re


In [17]:
# ----------------------------
# LOAD DATASETS
# ----------------------------

edu = pd.read_csv("EduVision_Final_Dataset__.csv")
ref = pd.read_csv("02_university_cleaned.csv")


In [18]:
# ----------------------------
# STANDARDIZE MISSING VALUES
# ----------------------------

missing_values = [
    -1, "-1",
    "NA", "N/A",
    "NULL", "null",
    "Unknown", "unknown",
    "", " "
]

edu.replace(missing_values, np.nan, inplace=True)
ref.replace(missing_values, np.nan, inplace=True)

In [19]:
# ----------------------------
# CLEAN UNIVERSITY NAMES
# ----------------------------

def clean_name(name):
    if pd.isna(name):
        return np.nan

    name = str(name).lower().strip()

    name = re.sub(r'[^a-z0-9 ]', '', name)

    return " ".join(name.split())

edu["merge_name"] = edu["University"].apply(clean_name)
ref["merge_name"] = ref["university_name"].apply(clean_name)


In [20]:
# ----------------------------
# KEEP LATEST RECORD
# ----------------------------

ref = (
    ref.sort_values("year")
       .groupby("merge_name")
       .tail(1)
)

In [21]:
# ----------------------------
# MERGE
# ----------------------------

merged = edu.merge(
    ref,
    on="merge_name",
    how="left",
    suffixes=("", "_ref")
)


In [22]:
# ----------------------------
# FILL MISSING VALUES
# ----------------------------

column_map = {

    "Publication_Count":"publications_count",
    "Citations_Count":"citations_count",
    "H_Index":"h_index",
    "Research_Output_Score":"research_output_score",
    "Research_Productivity_Index":"research_productivity_index",
    "Citation_Score":"citations_score",

    "Faculty_Count":"faculty_count",
    "Student_Faculty_Ratio":"faculty_to_student_ratio",

    "Enrollment":"total_students",

    "Undergrad_Students":"undergraduate_count",
    "Postgrad_Students":"postgraduate_count",

    "International_Students (%)":"international_student_ratio",

    "female_percentage":"female_percentage_ref",
    "male_percentage":"male_percentage_ref",

    "City":"city",
    "Region":"region",

    "type":"university_type",

    "QS_Rank":"world_rank",

    "Country_Avg_International_Ratio":
        "country_avg_international_ratio"
}

for target, source in column_map.items():

    if target in merged.columns and source in merged.columns:

        merged[target] = merged[target].fillna(
            merged[source]
        )

In [25]:
# ==========================
# COUNTRY MEDIAN FILL
# ==========================

numeric_cols = [
    "Publication_Count",
    "Citations_Count",
    "H_Index",
    "Research_Output_Score",
    "Research_Productivity_Index",
    "Citation_Score",
    "Faculty_Count",
    "Student_Faculty_Ratio",
    "Enrollment",
    "Undergrad_Students",
    "Postgrad_Students",
    "International_Students (%)"
]

for col in numeric_cols:

    if col not in merged.columns:
        continue

    # Convert to numeric
    merged[col] = (
        merged[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )

    merged[col] = pd.to_numeric(
        merged[col],
        errors="coerce"
    )

    # Country-wise median fill
    merged[col] = merged.groupby("Country")[col].transform(
        lambda x: x.fillna(x.median())
    )

    # Remaining nulls -> global median
    merged[col] = merged[col].fillna(
        merged[col].median()
    )

print("Country median fill completed.")

Country median fill completed.


In [26]:
# ----------------------------
# GLOBAL MEDIAN BACKUP
# ----------------------------

for col in numeric_cols:

    if col in merged.columns:

        merged[col] = merged[col].fillna(
            merged[col].median()
        )

In [27]:
# ----------------------------
# CITY / REGION / TYPE
# ----------------------------

for col in ["City","Region","type"]:

    if col in merged.columns:

        mode_value = merged[col].mode()

        if len(mode_value):

            merged[col] = merged[col].fillna(
                mode_value[0]
            )

In [28]:

# ----------------------------
# REMOVE TEMP COLUMNS
# ----------------------------

drop_cols = [c for c in merged.columns
             if c.endswith("_ref")]

merged.drop(
    columns=drop_cols,
    inplace=True,
    errors="ignore"
)

In [29]:

# ----------------------------
# CHECK MISSING %
# ----------------------------

missing = (
    merged.isnull().sum()
    / len(merged)
    * 100
).sort_values(ascending=False)

print(missing.head(30))

Alumni Employment              43.851852
Longitude                      43.851852
Quality of Faculty             43.851852
Influence                      43.851852
Latitude                       43.851852
Quality of Education           43.851852
Quality_Publications           43.851852
universities_ranked_count_y     6.370370
Female to Male Ratio            6.222222
Subject_Field                   4.740741
International Faculty Rank      4.444444
International Faculty Score     4.444444
2023 RANK                       4.148148
Sustainability Score            3.703704
Sustainability Rank             3.703704
RES.                            3.555556
world_rank                      2.962963
national_rank                   2.962963
country_avg_overall_score       2.814815
country_avg_rank                2.814815
universities_ranked_count       2.814815
university_name                 2.814815
year                            2.814815
gender_ratio                    2.814815
degree_level    

In [30]:
# ----------------------------
# SAVE
# ----------------------------

merged.to_csv(
    "EduVision_Final_Cleaned.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [31]:
# Missing percentage
missing_percent = (
    df.isnull().sum() / len(df) * 100
)

# Important columns to preserve
important_cols = [

    'University',
    'Country',
    'Region',

    'QS_Rank',
    'National_Rank',
    'Overall_Score',

    'Academic_Reputation',
    'Employer_Reputation',

    'Publication_Count',
    'Citations_Count',
    'H_Index',

    'Research_Output_Score',
    'Research_Productivity_Index',
    'Citation_Score',

    'Enrollment',
    'Faculty_Count',
    'Student_Faculty_Ratio',

    'Undergrad_Students',
    'Postgrad_Students',

    'International_Students (%)',

    'female_percentage',
    'male_percentage',

    'country_avg_rank',
    'country_avg_citations',
    'country_avg_overall_score',
    'country_avg_international_ratio'
]

# Find columns >2% missing
drop_cols = []

for col in df.columns:

    if (
        missing_percent[col] > 2
        and col not in important_cols
    ):
        drop_cols.append(col)

print("Columns to Drop:")
print(drop_cols)

df.drop(columns=drop_cols, inplace=True)

print("Final Shape:", df.shape)

Columns to Drop:
['2023 RANK', 'RES.', 'International Faculty Score', 'International Faculty Rank', 'International Students Score', 'International Students Rank', 'Sustainability Score', 'Sustainability Rank', 'Female to Male Ratio', 'Quality of Education', 'Alumni Employment', 'Quality of Faculty', 'Quality_Publications', 'Influence', 'Latitude', 'Longitude', 'type', 'Country_Avg_Citations', 'Subject_Field', 'City', 'universities_ranked_count_y']
Final Shape: (668, 66)


In [32]:
drop_cols = [
    'Quality of Education',
    'Alumni Employment',
    'Quality of Faculty',
    'Quality_Publications',
    'Influence',
    'universities_ranked_count_y'
]

df.drop(columns=drop_cols, inplace=True, errors='ignore')

print("New Shape:", df.shape)

# Check remaining missing %
missing_percent = (
    df.isnull().sum() / len(df) * 100
).sort_values(ascending=False)

print(missing_percent.head(20))

New Shape: (668, 66)
QS_Rank                                 53.892216
Region                                   7.185629
Citation_Score                           4.790419
Citations per Faculty Score              1.497006
Employment Outcomes Rank                 1.497006
Faculty Student Rank                     1.497006
Citations per Faculty Rank               1.497006
Faculty Student Score                    1.497006
male_percentage                          0.149701
university_id                            0.000000
2024 RANK                                0.000000
STATUS                                   0.000000
AGE                                      0.000000
FOCUS                                    0.000000
Academic Reputation Score                0.000000
University                               0.000000
Country                                  0.000000
Country Code                             0.000000
International Research Network Score     0.000000
International Research Networ

In [33]:
# Standardize university names
def clean_name(name):
    name = str(name).lower().strip()
    name = re.sub(r'[^a-z0-9 ]', '', name)
    return " ".join(name.split())

edu['merge_name'] = edu['University'].apply(clean_name)
ref['merge_name'] = ref['university_name'].apply(clean_name)

# Keep latest record per university
ref = (
    ref.sort_values('year')
       .groupby('merge_name')
       .tail(1)
)

# Merge world_rank
edu = edu.merge(
    ref[['merge_name', 'world_rank']],
    on='merge_name',
    how='left'
)

# Fill missing QS_Rank only
edu['QS_Rank'] = edu['QS_Rank'].fillna(edu['world_rank'])

# Cleanup
edu.drop(columns=['merge_name', 'world_rank'], inplace=True)

# Check remaining missing values
print("Missing QS_Rank:", edu['QS_Rank'].isna().sum())

# Save
edu.to_csv("EduVision_Final_Cleaned.csv", index=False)

Missing QS_Rank: 2


In [34]:
# Check remaining missing %
missing_percent = (
    df.isnull().sum() / len(df) * 100
).sort_values(ascending=False)

print(missing_percent.head(20))

QS_Rank                                 53.892216
Region                                   7.185629
Citation_Score                           4.790419
Citations per Faculty Score              1.497006
Employment Outcomes Rank                 1.497006
Faculty Student Rank                     1.497006
Citations per Faculty Rank               1.497006
Faculty Student Score                    1.497006
male_percentage                          0.149701
university_id                            0.000000
2024 RANK                                0.000000
STATUS                                   0.000000
AGE                                      0.000000
FOCUS                                    0.000000
Academic Reputation Score                0.000000
University                               0.000000
Country                                  0.000000
Country Code                             0.000000
International Research Network Score     0.000000
International Research Network Rank      0.000000


In [35]:
import pandas as pd
import re

edu = pd.read_csv("EduVision_Final_Cleaned.csv")
ref = pd.read_csv("02_university_cleaned.csv")

def clean_name(name):
    name = str(name).lower().strip()
    name = re.sub(r'[^a-z0-9 ]', '', name)
    return " ".join(name.split())

edu['merge_name'] = edu['University'].apply(clean_name)
ref['merge_name'] = ref['university_name'].apply(clean_name)

print("Edu universities:", edu['merge_name'].nunique())
print("Reference universities:", ref['merge_name'].nunique())

matches = edu['merge_name'].isin(ref['merge_name'])

print("Matched Universities:", matches.sum())
print("Unmatched Universities:", (~matches).sum())

Edu universities: 668
Reference universities: 3334
Matched Universities: 656
Unmatched Universities: 19


In [36]:
print(ref.columns.tolist())

['university_id', 'university_name', 'year', 'world_rank', 'national_rank', 'overall_score', 'country', 'region', 'city', 'university_type', 'academic_reputation_score', 'employer_reputation_score', 'citations_score', 'publications_count', 'citations_count', 'citations_per_faculty', 'h_index', 'research_output_score', 'research_productivity_index', 'subject_field', 'total_students', 'international_students_count', 'international_student_ratio', 'faculty_count', 'faculty_to_student_ratio', 'gender_ratio', 'female_percentage', 'male_percentage', 'degree_level', 'undergraduate_count', 'postgraduate_count', 'country_avg_rank', 'universities_ranked_count', 'best_university_rank', 'country_avg_overall_score', 'country_avg_academic_reputation', 'country_avg_citations', 'country_avg_international_ratio', 'merge_name']


In [37]:
unmatched = edu.loc[
    ~edu['merge_name'].isin(ref['merge_name']),
    ['University']
]

print(unmatched.head(50))

                                            University
42                    National Taiwan University (NTU)
141              National Cheng Kung University (NCKU)
164  Ulsan National Institute of Science and Techno...
176              Bandung Institute of Technology (ITB)
193  Daegu Gyeongbuk Institute of Science and Techn...
205  Gwangju Institute of Science and Technology (G...
241  National Taiwan University of Science and Tech...
345         Tokyo Medical and Dental University (TMDU)
376                      Universidad Panamericana (UP)
409              Instituto PolitÃ©cnico Nacional (IPN)
420                Universiti Tenaga Nasional (UNITEN)
432  University of Engineering & Technology (UET) L...
450               Universiti Tunku Abdul Rahman (UTAR)
496  Samara National Research University (Samara Un...
570                    Universiti Malaysia Sabah (UMS)
571               Universiti Malaysia Sarawak (UNIMAS)
572               Universiti Malaysia Terengganu (UMT)
632    Uni

In [38]:
region_mode = (
    edu.groupby('Country')['Region']
       .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
)

edu['Region'] = edu.apply(
    lambda row:
    region_mode[row['Country']]
    if pd.isna(row['Region'])
    else row['Region'],
    axis=1
)

In [39]:
import pandas as pd
import re

edu = pd.read_csv("EduVision_Final_Cleaned.csv")
ref = pd.read_csv("02_university_cleaned.csv")

def clean_name(name):
    name = str(name).lower().strip()
    name = re.sub(r'[^a-z0-9 ]', '', name)
    return " ".join(name.split())

edu["merge_name"] = edu["University"].apply(clean_name)
ref["merge_name"] = ref["university_name"].apply(clean_name)

matched = edu["merge_name"].isin(ref["merge_name"])

print("Total Universities:", len(edu))
print("Matched:", matched.sum())
print("Unmatched:", (~matched).sum())
print("Match %:", round(matched.mean()*100,2))

Total Universities: 675
Matched: 656
Unmatched: 19
Match %: 97.19


In [40]:
unmatched = edu.loc[
    ~edu["merge_name"].isin(ref["merge_name"]),
    ["University"]
]

print(unmatched.head(50))

                                            University
42                    National Taiwan University (NTU)
141              National Cheng Kung University (NCKU)
164  Ulsan National Institute of Science and Techno...
176              Bandung Institute of Technology (ITB)
193  Daegu Gyeongbuk Institute of Science and Techn...
205  Gwangju Institute of Science and Technology (G...
241  National Taiwan University of Science and Tech...
345         Tokyo Medical and Dental University (TMDU)
376                      Universidad Panamericana (UP)
409              Instituto PolitÃ©cnico Nacional (IPN)
420                Universiti Tenaga Nasional (UNITEN)
432  University of Engineering & Technology (UET) L...
450               Universiti Tunku Abdul Rahman (UTAR)
496  Samara National Research University (Samara Un...
570                    Universiti Malaysia Sabah (UMS)
571               Universiti Malaysia Sarawak (UNIMAS)
572               Universiti Malaysia Terengganu (UMT)
632    Uni

In [41]:
import re

def clean_name(name):
    name = str(name).lower()

    # remove text inside brackets
    name = re.sub(r'\([^)]*\)', '', name)

    # replace &
    name = name.replace('&', 'and')

    # remove punctuation
    name = re.sub(r'[^a-z0-9 ]', ' ', name)

    # remove extra spaces
    name = ' '.join(name.split())

    return name

In [42]:
edu["merge_name"] = edu["University"].apply(clean_name)
ref["merge_name"] = ref["university_name"].apply(clean_name)

matched = edu["merge_name"].isin(ref["merge_name"])

print("Matched:", matched.sum())
print("Unmatched:", (~matched).sum())
print("Match %:", round(matched.mean()*100,2))

Matched: 675
Unmatched: 0
Match %: 100.0


In [44]:
dup = ref[ref.duplicated("merge_name", keep=False)]

print("Duplicate merge_names:", dup["merge_name"].nunique())
print(dup[["university_name","year","merge_name"]].head(20))

Duplicate merge_names: 2878
                             university_name  year  \
1                   AGH University of Krakow  2016   
2                   AGH University of Krakow  2017   
3                   AGH University of Krakow  2018   
4                   AGH University of Krakow  2019   
5                   AGH University of Krakow  2020   
6                   AGH University of Krakow  2021   
7                   AGH University of Krakow  2022   
8                   AGH University of Krakow  2023   
9                   AGH University of Krakow  2024   
10                  AGH University of Krakow  2025   
11                  AGH University of Krakow  2026   
12  AGH University of Science and Technology  2018   
13  AGH University of Science and Technology  2019   
14  AGH University of Science and Technology  2020   
15  AGH University of Science and Technology  2021   
16  AGH University of Science and Technology  2022   
17  AGH University of Science and Technology  2023   


In [45]:
ref_unique = (
    ref.sort_values("year")
       .drop_duplicates("merge_name", keep="last")
)

rank_map = ref_unique.set_index("merge_name")["world_rank"]

edu["QS_Rank"] = edu["QS_Rank"].fillna(
    edu["merge_name"].map(rank_map)
)

In [46]:
ref_unique = ref.groupby("merge_name", as_index=False).agg({
    "world_rank":"min"
})

rank_map = ref_unique.set_index("merge_name")["world_rank"]

edu["QS_Rank"] = edu["QS_Rank"].fillna(
    edu["merge_name"].map(rank_map)
)

In [47]:
print(
    "Missing QS_Rank:",
    edu["QS_Rank"].isna().sum()
)

print(
    "Missing %:",
    round(
        edu["QS_Rank"].isna().mean()*100,
        2
    )
)

Missing QS_Rank: 0
Missing %: 0.0


In [48]:
print(
    ref["merge_name"].duplicated().sum()
)

18703


In [49]:
# Keep latest year for each university
ref_unique = (
    ref.sort_values("year")
       .drop_duplicates(subset=["merge_name"], keep="last")
)

print("Unique universities:", len(ref_unique))

Unique universities: 3312


In [50]:
print(ref_unique["merge_name"].duplicated().sum())

0


In [51]:
rank_map = ref_unique.set_index("merge_name")["world_rank"]

edu["QS_Rank"] = edu["QS_Rank"].fillna(
    edu["merge_name"].map(rank_map)
)

In [52]:
missing_percent = (
    edu["QS_Rank"].isnull().sum() / len(edu)
) * 100

print(f"QS_Rank Missing %: {missing_percent:.2f}%")

QS_Rank Missing %: 0.00%


In [53]:
missing_percent = (
    edu["QS_Rank"].isnull().sum() / len(edu)
) * 100

print(f"QS_Rank Missing %: {missing_percent:.2f}%")

QS_Rank Missing %: 0.00%


In [54]:
region_map = (
    df.groupby("Country")["Region"]
      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
)

df["Region"] = df.apply(
    lambda row:
    region_map[row["Country"]]
    if pd.isna(row["Region"])
    else row["Region"],
    axis=1
)

In [55]:
df["Citation_Score"] = (
    df.groupby("Country")["Citation_Score"]
      .transform(lambda x: x.fillna(x.median()))
)

df["Citation_Score"] = df["Citation_Score"].fillna(
    df["Citation_Score"].median()
)

In [56]:
small_missing_cols = [
    "Citations per Faculty Score",
    "Employment Outcomes Rank",
    "Faculty Student Rank",
    "Faculty Student Score"
]

for col in small_missing_cols:
    df[col] = df[col].fillna(df[col].median())

TypeError: Cannot convert ['45' '2' '3' '1' '6' '126' '5' '47' '46' '18' '10' '9' '11' '29' '60'
 '27' '22' '33' '19' '134' '118' '131' '8' '42' '194' '99' '163' '35'
 '330' '113' '37' '269' '100' '133' '170' '95' '59' '181' '73' '12' '111'
 '58' '182' '306' '71' '361' '61' '235' '48' '98' '486' '84' '274' '74'
 '224' '83' '175' '270' '370' '193' '82' '262' '335' '275' '173' '342'
 '86' '246' '439' '76' '32' '39' '303' '506' '509' '313' '278' '438' '105'
 '426' '152' '372' '701+' '356' '400' '260' '349' '547' '179' '197' '124'
 '201' '665' '57' '112' '421' '442' '109' '308' '156' '336' '153' '176'
 '357' '301' '354' '310' '259' '30' '101' '75' '184' '701+' '290' '525'
 '151' '621' '324' '687' '65' '492' '159' '320' '358' '167' '50' '237'
 '87' '379' '408' '277' '414' '464' '701+' '44' '206' '155' '106' '24'
 '218' '38' '251' '344' '446' '701+' '268' '188' '149' '412' '468' '667'
 '43' '247' '257' '62' '701+' '508' '701+' '282' '294' '160' '229' '392'
 '503' '701+' '385' '143' '67' '629

In [57]:
rank_cols = [
    "Employment Outcomes Rank",
    "Faculty Student Rank",
    "Citations per Faculty Rank",
    "International Faculty Rank",
    "International Students Rank"
]

for col in rank_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("701+", "701", regex=False)
        )

        df[col] = pd.to_numeric(df[col], errors="coerce")

        df[col] = df[col].fillna(df[col].median())

In [58]:
score_cols = [
    "Faculty Student Score",
    "Citations per Faculty Score",
    "Citation_Score"
]

for col in score_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())

In [59]:
missing_percent = (
    df.isnull().sum() / len(df) * 100
).sort_values(ascending=False)

print(missing_percent.head(20))

QS_Rank                                53.892216
Region                                  0.598802
male_percentage                         0.149701
university_id                           0.000000
2024 RANK                               0.000000
University                              0.000000
FOCUS                                   0.000000
Country Code                            0.000000
Country                                 0.000000
SIZE                                    0.000000
Academic Reputation Rank                0.000000
Employer Reputation Score               0.000000
Employer Reputation Rank                0.000000
Faculty Student Score                   0.000000
Faculty Student Rank                    0.000000
AGE                                     0.000000
STATUS                                  0.000000
Academic Reputation Score               0.000000
International Research Network Rank     0.000000
Employment Outcomes Score               0.000000
dtype: float64


In [60]:
print("edu missing:", edu["QS_Rank"].isna().sum())

print("df missing:", df["QS_Rank"].isna().sum())

edu missing: 0
df missing: 360


In [61]:
df["QS_Rank"] = edu["QS_Rank"]

In [62]:
print(
    round(
        df["QS_Rank"].isna().mean()*100,
        2
    )
)

0.0


In [63]:
for col in df.columns:
    if df[col].dtype == 'object':
        sample = df[col].astype(str)
        if sample.str.contains(r'\d+\+', regex=True, na=False).any():
            print(col)

2024 RANK
Academic Reputation Rank
Employer Reputation Rank
International Research Network Rank


In [64]:
drop_cols = [
    'Academic Reputation Rank',
    'Employer Reputation Rank',
    'International Research Network Rank'
]

df.drop(columns=drop_cols, inplace=True, errors='ignore')

print("New Shape:", df.shape)

New Shape: (668, 63)


In [65]:
missing_percent = (
    df.isnull().sum() / len(df) * 100
).sort_values(ascending=False)

print(missing_percent.head(20))

Region                                  0.598802
male_percentage                         0.149701
university_id                           0.000000
Country Code                            0.000000
Country                                 0.000000
2024 RANK                               0.000000
University                              0.000000
AGE                                     0.000000
STATUS                                  0.000000
Academic Reputation Score               0.000000
Employer Reputation Score               0.000000
Faculty Student Score                   0.000000
Faculty Student Rank                    0.000000
SIZE                                    0.000000
FOCUS                                   0.000000
International Research Network Score    0.000000
Employment Outcomes Score               0.000000
Employment Outcomes Rank                0.000000
Student Population                      0.000000
International Students                  0.000000
dtype: float64


In [66]:
edu["QS_Rank"] = edu["QS_Rank"].fillna(
    edu["merge_name"].map(rank_map)
)

In [67]:
df["QS_Rank"].astype(str).str.contains(r"\+", na=False).sum()

np.int64(0)

In [68]:
print(df["QS_Rank"].head())
print(df["QS_Rank"].dtype)

4    2
3    3
1    4
2    5
0    6
Name: QS_Rank, dtype: object
object


In [69]:
# 1. Shape
print("Shape:", df.shape)

# 2. Duplicate rows
print("Duplicate Rows:", df.duplicated().sum())

# 3. Missing values %
missing_percent = (
    df.isnull().sum() / len(df) * 100
).sort_values(ascending=False)

print("\nTop Missing Columns:")
print(missing_percent.head(20))

# 4. Check rank columns with '+' remaining
for col in df.columns:
    if df[col].dtype == "object":
        plus_count = df[col].astype(str).str.contains(r"\+", na=False).sum()
        if plus_count > 0:
            print(f"{col}: {plus_count} values containing '+'")

# 5. Check important columns
important_cols = [
    'University',
    'Country',
    'QS_Rank',
    'Region'
]

print("\nMissing in Important Columns:")
for col in important_cols:
    if col in df.columns:
        print(col, ":", df[col].isnull().sum())

Shape: (668, 63)
Duplicate Rows: 0

Top Missing Columns:
Region                                  0.598802
male_percentage                         0.149701
university_id                           0.000000
Country Code                            0.000000
Country                                 0.000000
2024 RANK                               0.000000
University                              0.000000
AGE                                     0.000000
STATUS                                  0.000000
Academic Reputation Score               0.000000
Employer Reputation Score               0.000000
Faculty Student Score                   0.000000
Faculty Student Rank                    0.000000
SIZE                                    0.000000
FOCUS                                   0.000000
International Research Network Score    0.000000
Employment Outcomes Score               0.000000
Employment Outcomes Rank                0.000000
Student Population                      0.000000
Internationa

In [73]:
df.to_csv("University_Cleaned.csv", index=False)
print("Saved successfully")

Saved successfully
